# Copilot

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException

from bs4 import BeautifulSoup
import pandas as pd
import re
import numpy as np
from datetime import timedelta
from sklearn.preprocessing import MinMaxScaler

# Scrape data

In [181]:
# Open Chrome webdriver
driver = webdriver.Chrome()
url = "https://www.ocasionplus.com/coches-ocasion/v2?orderBy=lowerPrice&kms%5Bto%5D=100000&year%5Bfrom%5D=2016&year%5Bto%5D=2022&environment=0_EMISIONES%2CC%2CECO&power%5Bfrom%5D=90&price%5Bto%5D=16000"

# Testing URLS
# url = "https://www.ocasionplus.com/coches-ocasion/v2?orderBy=lowerPrice&kms%5Bto%5D=35000&year%5Bto%5D=2021&doors=5&environment=0_EMISIONES%2CC%2CECO&power%5Bfrom%5D=80"
# url = "https://www.ocasionplus.com/coches-ocasion/v2?orderBy=lowerPrice&kms%5Bto%5D=10000&year%5Bto%5D=2021&doors=5&power%5Bfrom%5D=80"

driver.get(url)

# Scroll until seeMoreCars_button disappears
WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, "seeMoreCars_button__WXdaL")))
while True:
    try:
        driver.execute_script("arguments[0].scrollIntoView();", driver.find_elements(By.CLASS_NAME, "cardVehicle_card__LwFCi")[-1])
        WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.CSS_SELECTOR, ".seeMoreCars_button__WXdaL")))
    except NoSuchElementException:
        print('no such element')
        break
    except TimeoutException:
        # print('timeout')
        break

html = driver.page_source
driver.quit()

soup = BeautifulSoup(html, 'html.parser')
html = soup.find_all('div', class_='cardVehicle_card__LwFCi')

timeout


# Parse scraped data

In [221]:
cars_data = []
lens = []
for car in html:
    discount = car.find('p', class_='topLayer_discount__8zFJc')
    if discount is None:
        status = 'Available'
    else:
        discount = car.find('p', class_='topLayer_discount__8zFJc').text.capitalize()
        if discount == 'Reservado':
            status = 'Reservado'
            discount = None
        elif discount == 'Vendido':
            status = 'Vendido'
            discount = None
        else:
            status = 'Available'
            discount = int(re.sub(r'\D', '', discount))
    
    title = car.find_all('span', class_='cardVehicle_spot__e6YZx')
    if len(title) == 2:
        full_price = title[1].text
        full_price = int(re.sub(r'\D', '', full_price))
    else:
        full_price = None
    title = title[0].text

    description = car.find_all('span', class_='cardVehicle_finance__SG6JV')
    discounted_price = description[1].text
    discounted_price = int(re.sub(r'\D', '', discounted_price))
    description = description[0].text

    monthly_rate = car.find('span', class_='cardVehicle_cuote__eSQXl').text
    monthly_rate = int(re.sub(r'\D', '', monthly_rate))

    details = car.find_all('span', class_='characteristics_elements__Mb1S_')
    details = [detail.text for detail in details]
    year = int(details[0])
    mileage = details[1]
    mileage = int(re.sub(r'\D', '', mileage))
    fuel = details[2]
    cv = details[3]
    cv = int(re.sub(r'\D', '', cv))
    transmission = details[4].capitalize()
    try:
        if details[5] == 'Vista 360º':
            vista = details[5]
        elif details[5] == 'Único propietario':
            owner = details[5]
        elif details[5] == 'Garantía del fabricante':
            warranty = details[5]
        elif details[5] == 'Libro de revisiones':
            libro = details[5]
        else:
            vista = None
    except IndexError:
        vista = None
    try:
        if details[6] == 'Único propietario':
            owner = details[6]
        elif details[6] == 'Garantía del fabricante':
            warranty = details[6]
        elif details[6] == 'Libro de revisiones':
            libro = details[6]
        else:
            owner = None
    except IndexError:
        owner = None
    try:
        if details[7] == 'Garantía del fabricante':
            warranty = details[7]
        elif details[7] == 'Libro de revisiones':
            libro = details[7]
        else:
            warranty = None
    except IndexError:
        warranty = None
    try:
        libro = details[8]
    except IndexError:
        libro = None

    city = car.find('div', class_='cardVehicle_dealer__41Xs5').text.split('-')
    if len(city) == 2:
        district = city[1]
        city = city[0]
    else:
        district = None
        city = city[0]
    url = car.find('a', class_="cardVehicle_link__l8xYT")['href']
    url = 'https://www.ocasionplus.com' + url

    cars_data.append([discount, status, title, full_price, description, discounted_price, monthly_rate, year, mileage, fuel, cv, transmission, vista, owner, warranty, libro, city, district, url])
    
df = pd.DataFrame(cars_data, columns=["discount", "status", "title", "full_price", "description", "discounted_price", "monthly_rate", "year", "mileage", "fuel", "cv", "transmission", "vista", "owner", "warranty", "libro", 'city', 'district', 'url'])

# df = scrape_brands(brands)
df.to_csv('ocasion_cars_raw.csv', index=False)

REMEMBER to clean up the data in excel, and export the refined data
- discount and full_price values will be afterscraping

# Get location commute time

In [216]:
keys = ['Huelva', 'Cádiz', 'Murcia', 'Valencia', 'Barcelona', 'Madrid',
       'Sevilla', 'Castellón', 'Córdoba', 'Jaén', 'Zaragoza', 'Santander',
       'Vizcaya', 'Granada', 'Navarra', 'Málaga', 'Almería', 'Badajoz',
       'Alicante', 'Toledo', 'Gipuzkoa', 'Ciudad Real', 'Girona',
       'La Coruña', 'Pontevedra', 'Albacete', 'Oviedo', 'Cáceres']
values = [timedelta(hours=3, minutes=57), timedelta(hours=4, minutes=5), timedelta(hours=2, minutes=45), timedelta(hours=1, minutes=40), timedelta(hours=2, minutes=20), timedelta(hours=0, minutes=0), timedelta(hours=2, minutes=20),timedelta(hours=2, minutes=59), timedelta(hours=1, minutes=40), timedelta(hours=3, minutes=51), timedelta(hours=1, minutes=15), timedelta(hours=3, minutes=55), timedelta(hours=3, minutes=46), timedelta(hours=3, minutes=17), timedelta(hours=3, minutes=32), timedelta(hours=2, minutes=30), timedelta(hours=5, minutes=58), timedelta(hours=4, minutes=17), timedelta(hours=2, minutes=30), timedelta(hours=0, minutes=33), timedelta(hours=5, minutes=56), timedelta(hours=0, minutes=54), timedelta(hours=3, minutes=18), timedelta(hours=3, minutes=48), timedelta(hours=3, minutes=54), timedelta(hours=1, minutes=31), timedelta(hours=3, minutes=18), timedelta(hours=3, minutes=7)]

distance_time = dict(zip(keys, values))

In [270]:
df = pd.read_csv(r'data\Used cars - ocasion_cars_raw.csv')
df = df[df.status == 'Available']
df['travel_time'] = df['city'].map(distance_time)

# Normalize data

In [269]:
df2 = pd.read_csv(r'data\Used cars - ocasion_cars_raw.csv')
df2 = df2[df2.status == 'Available']
df2 = df2.drop('district', axis=1)
df2 = df2.drop('discount', axis=1)
df2 = df2.drop('status', axis=1)
df2 = df2.drop('year', axis=1)
df2 = df2.drop('monthly_rate', axis=1)
df2['travel_time'] = df2['city'].map(distance_time)
df2 = df2.drop('city', axis=1)

# Codify
fuel_code = {'Diésel': 0, 'Gasolina': 1, 'Híbrido': 2, 'GLP': 3, 'GNC': 3, 'Eléctrico': 3}
df2['fuel'] = df2['fuel'].apply(lambda x: fuel_code.get(x, 0))

df2['vista'] = df2['vista'].notna().astype(int)
df2['owner'] = df2['owner'].notna().astype(int)
df2['warranty'] = df2['warranty'].notna().astype(int)
df2['libro'] = df2['libro'].notna().astype(int)
df2['travel_time'] = df2['travel_time'] / df2['travel_time'].max()


# Normalize to min
scaler = MinMaxScaler()

df2['% discount'] = scaler.fit_transform(df2[['% discount']])
df2['discounted_price'] = scaler.fit_transform(df2[['discounted_price']])
df2['full_price'] = scaler.fit_transform(df2[['full_price']])
df2['age'] = scaler.fit_transform(df2[['age']])
df2['mileage'] = scaler.fit_transform(df2[['mileage']])
df2['fuel'] = scaler.fit_transform(df2[['fuel']])
df2['cv'] = scaler.fit_transform(df2[['cv']])

transmission_code = {'Auto': 1, 'Manual': 0}
df2['transmission'] = df2['transmission'].apply(lambda x: transmission_code.get(x, 0))

# Correct valence
df2['discounted_price'] = (df2['discounted_price'] - 1)* -1
df2['full_price'] = (df2['full_price'] - 1)* -1
df2['age'] = (df2['age'] - 1)* -1
df2['mileage'] = (df2['mileage'] - 1)* -1
df2['travel_time'] = (df2['travel_time'] - 1)* -1


### Export with user adjustment

In [271]:
# Rated CSV
users = pd.read_csv(r'data\Used cars - Sheet2.csv')
users = users.drop('type', axis=1)
users = users.set_index('name')
users = users.squeeze()
users['full_price'] = users['discounted_price']

# Apply rating and fix NaN values created by applying values
df3 = df2.mul(users)
df3['title'] = df2['title']
df3['description'] = df2['description']
df3['url'] = df2['url']

# Re-organize columns
df3 = df3[['% discount', 'discounted_price', 'full_price', 'age', 'mileage', 'cv', 'fuel',
       'libro', 'owner', 'travel_time', 'transmission',
       'vista', 'warranty', 'title', 'description','url']]

# # Add score column
# df3['score'] = df3['% discount'] + df3['full_price'] + df3['age'] + df3['mileage'] + df3['cv'] + df3['fuel'] + df3['libro'] + df3['owner'] + df3['travel_time'] + df3['transmission'] + df3['vista'] + df3['warranty']

df3.to_csv('ocasion_cars.csv', index=False)

### Export without user adjustment
The rating data is at the bottom of the csv

In [267]:
# No rating CSV
users = pd.read_csv(r'data\Used cars - Sheet2.csv')
users = users.drop('type', axis=1)
users = users.set_index('name')
users = users.squeeze()
users = users.to_frame(name=users.name).reindex(df2.columns)
users = users.transpose()
users.full_price = users.discounted_price

df2.to_csv('ocasion_cars_norating.csv', index=False)